# Code agent lấy thời tiết Hà Nội

In [ ]:
import requests
import os
from dotenv import load_dotenv
from smolagents import CodeAgent, InferenceClientModel, FinalAnswerTool, tool

load_dotenv()

@tool
def get_coordinates(city_name: str) -> dict:
    """
    Lấy tọa độ địa lý (latitude, longitude) của một thành phố.
    
    Args:
        city_name: Tên thành phố cần tra cứu (ví dụ: 'Hanoi', 'Ho Chi Minh City')
    
    Returns:
        Một dict chứa 'latitude', 'longitude' và 'name' của thành phố.
    """
    url = "https://geocoding-api.open-meteo.com/v1/search"
    response = requests.get(url, params={"name": city_name, "count": 1, "language": "vi"})
    data = response.json()
    if not data.get("results"):
        return {"error": f"Không tìm thấy thành phố: {city_name}"}
    result = data["results"][0]
    return {
        "name": result["name"],
        "latitude": result["latitude"],
        "longitude": result["longitude"]
    }

@tool
def get_current_weather(latitude: float, longitude: float) -> dict:
    """
    Lấy thông tin thời tiết hiện tại theo tọa độ địa lý.
    
    Args:
        latitude: Vĩ độ của địa điểm.
        longitude: Kinh độ của địa điểm.
    
    Returns:
        Một dict chứa nhiệt độ, độ ẩm, tốc độ gió và mô tả thời tiết.
    """
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "current": ["temperature_2m", "relative_humidity_2m", "wind_speed_10m", "weather_code"],
        "timezone": "auto"
    }
    response = requests.get(url, params=params)
    data = response.json()
    current = data["current"]
    
    weather_codes = {
        0: "Trời quang", 1: "Chủ yếu quang", 2: "Có mây một phần", 3: "Nhiều mây",
        45: "Sương mù", 51: "Mưa phùn nhẹ", 61: "Mưa nhẹ", 63: "Mưa vừa", 65: "Mưa to",
        80: "Mưa rào nhẹ", 81: "Mưa rào vừa", 82: "Mưa rào nặng",
        95: "Dông", 99: "Dông kèm mưa đá"
    }
    
    return {
        "temperature": f"{current['temperature_2m']}°C",
        "humidity": f"{current['relative_humidity_2m']}%",
        "wind_speed": f"{current['wind_speed_10m']} km/h",
        "weather": weather_codes.get(current["weather_code"], f"Mã: {current['weather_code']}")
    }

agent = CodeAgent(
    tools=[get_coordinates, get_current_weather, FinalAnswerTool()],
    model=InferenceClientModel(token = os.getenv("HF_TOKEN"))
)

agent.run("Hãy cho tôi biết thời tiết hiện tại ở Hà Nội.")


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Hãy cho tôi biết thời tiết hiện tại ở Hà Nội.                                                                   │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen3-Next-80B-A3B-Thinking ───────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  coordinates = get_coordinates(city_name="Hanoi")                                                                 
  print(coordinates)                                                                                               
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
{'name': 'Hà Nội', 'latitude': 21.0245, 'longitude': 105.84117}

Out: None

[Step 1: Duration 17.48 seconds| Input tokens: 2,158 | Output tokens: 3,146]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer(f"Thời tiết Hà Nội hiện tại: Nhiệt độ {weather_data['temperature']}, độ ẩm                          
  {weather_data['humidity']}, gió {weather_data['wind_speed']}, {weather_data['weather']}")                        
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Thời tiết Hà Nội hiện tại: Nhiệt độ 26.0°C, độ ẩm 93%, gió 15.5 km/h, Trời quang

[Step 3: Duration 5.55 seconds| Input tokens: 6,938 | Output tokens: 5,678]

'Thời tiết Hà Nội hiện tại: Nhiệt độ 26.0°C, độ ẩm 93%, gió 15.5 km/h, Trời quang'

# Code agent tính số Fibonacci

In [ ]:
import os
from dotenv import load_dotenv
from smolagents import CodeAgent, DuckDuckGoSearchTool, InferenceClientModel

load_dotenv()

agent = CodeAgent(
    tools=[DuckDuckGoSearchTool()],
    model=InferenceClientModel(token = os.getenv("HF_TOKEN"))
)

agent.run("Số Fibonacci thứ 50 là bao nhiêu?")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Số Fibonacci thứ 50 là bao nhiêu?                                                                               │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen3-Next-80B-A3B-Thinking ───────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  a, b = 0, 1                                                                                                      
  for _ in range(50):                                                                                              
      a, b = b, a + b                                                                                              
  final_answer(a)                                                                                                  
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 12586269025

[Step 1: Duration 31.79 seconds| Input tokens: 2,087 | Output tokens: 4,163]

12586269025

# Tool calling agent lấy thời tiết Hà Nội

In [17]:
import os
from dotenv import load_dotenv
from smolagents import ToolCallingAgent, DuckDuckGoSearchTool, InferenceClientModel

load_dotenv()

agent = ToolCallingAgent(
    tools=[DuckDuckGoSearchTool()],
    model=InferenceClientModel(model_id="Qwen/Qwen2.5-72B-Instruct", token = os.getenv("HF_TOKEN"))
)

agent.run("Thời tiết hiện tại ở Hà Nội như thế nào?")


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Thời tiết hiện tại ở Hà Nội như thế nào?                                                                        │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-72B-Instruct ──────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'web_search' with arguments: {'query': 'thời tiết hiện tại ở Hà Nội'}                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: ## Search Results

|Thời tiết hàng giờ ở Hà Nội, Hà Nội, Việt Nam | 
AccuWeather](https://www.accuweather.com/vi/vn/hanoi/353412/hourly-weather-forecast/353412)
Check current conditions in Hà Nội, Hà Nội, Việt Nam with radar, hourly, and more.

|Thời tiết Hà Nội hôm nay, sắp tới có mưa không? - Thoitiet.vn](https://thoitiet.vn/ha-noi)
Cập nhật dự báo thời tiết Hà Nội hôm nay và ngày mai chính xác nhất. Dự báo khả năng có mưa, lượng mưa, thời tiết 
cực đoan và nhiệt độ ở Hà Nội

|Dự báo thời tiết Hà Nội hôm nay và nhiều ngày tới - xemthoitiet.com.vn](https://xemthoitiet.com.vn/ha-noi)
Theo dõi dự báo thời tiết Hà Nội chính xác nhất. Cập nhật nhiệt độ, lượng mưa, độ ẩm, gió và chỉ số UV hôm nay, 
ngày mai và nhiều ngày tới - xemthoitiet.com.vn.

|Dự báo thời tiết Hà Nội hàng giờ - Thoitiet24h.vn](https://thoitiet24h.vn/thoi-tiet-hang-gio/ha-noi)
Dự báo thời tiết Hà Nội hàng giờ bao nhiêu độ (℃), lượng mưa (mm), chỉ số UV, độ ẩm (%). Chi tiết thời tiết Hà Nội 
hàng giờ tại Thời tiết 24h.

|Thời tiết hàng giờ ở Hà nội, Hà nội, Việt Nam | 
Tomorrow.io](https://weather.tomorrow.io/vi/VN/HN/Hanoi/130201/hourly/)
Nhận dự báo thời tiết hàng giờ mới nhất ở Hà nội, bao gồm tất cả các điều kiện thời tiết mà bạn cần - nhiệt độ, 
lượng mưa, độ ẩm và gió - với dự báo của Tomorrow.io.

|Thời tiết - Hà Nội - Dự báo 14 ngày: Nhiệt độ, Gió & Radar](https://www.ventusky.com/vi/ha-ni)
Hà Nội - Dự báo thời tiết cho thời gian 14 ngày, thông tin từ các trạm khí tượng thủy văn, ảnh chụp từ các 
webcamera, mặt trời mọc và lặn, bản đồ gió và mưa cho khu vực.

|Dự báo thời tiết Hà Nội hôm nay & ngày mai - Cập nhật liên tục, nhanh 
chóng](https://xemthoitiet.vn/thoi-tiet/ha-noi/)
Xem dự báo thời tiết Hà Nội hôm nay và ngày mai chi tiết nhất. Cập nhật liên tục, chính xác giúp bạn chủ động kế 
hoạch mỗi ngày.

|Dự báo thời tiết Hà Nội hôm nay và ngày mai - Thoitiet360](https://thoitiet360.edu.vn/ha-noi)
Dự báo thời tiết Hà Nội hôm nay và ngày mai, những ngày tới chính xác. Cập nhật lượng mưa tại Hà Nội trên 
thoitiet360.edu.vn.

|Dự báo thời tiết Hà Nội hôm nay và những ngày tới](https://thoitiet247.vn/ha-noi)
Tình hình dự báo thời tiết, nhiệt độ ở Hà Nội hôm nay, ngày mai và các ngày tới. Dự báo khả năng có mưa hay không 
theo giờ chính xác nhất!

|Dự báo thời tiết Hà Nội - thoitiet.co](https://thoitiet.co/ha-noi/)
Dự báo thời tiết ở Hà Nội hôm nay và 7 ngày tới: Nhiệt độ, lượng mưa, độ ẩm được cập nhật liên tục chính xác theo 
giờ.

[Step 1: Duration 4.04 seconds| Input tokens: 1,145 | Output tokens: 24]

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'Thời tiết hiện tại ở Hà Nội đang thay đổi theo từng    │
│ giờ, bạn có thể kiểm tra chi tiết tại                                                                           │
│ [AccuWeather](https://www.accuweather.com/vi/vn/hanoi/353412/hourly-weather-forecast/353412) hoặc               │
│ [Thoitiet.vn](https://thoitiet.vn/ha-noi) để biết thêm thông tin về nhiệt độ, lượng mưa, độ ẩm và gió.'}        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: Thời tiết hiện tại ở Hà Nội đang thay đổi theo từng giờ, bạn có thể kiểm tra chi tiết tại 
|AccuWeather](https://www.accuweather.com/vi/vn/hanoi/353412/hourly-weather-forecast/353412) hoặc 
|Thoitiet.vn](https://thoitiet.vn/ha-noi) để biết thêm thông tin về nhiệt độ, lượng mưa, độ ẩm và gió.

Final answer: Thời tiết hiện tại ở Hà Nội đang thay đổi theo từng giờ, bạn có thể kiểm tra chi tiết tại 
[AccuWeather](https://www.accuweather.com/vi/vn/hanoi/353412/hourly-weather-forecast/353412) hoặc 
[Thoitiet.vn](https://thoitiet.vn/ha-noi) để biết thêm thông tin về nhiệt độ, lượng mưa, độ ẩm và gió.

[Step 2: Duration 7.08 seconds| Input tokens: 3,155 | Output tokens: 141]

'Thời tiết hiện tại ở Hà Nội đang thay đổi theo từng giờ, bạn có thể kiểm tra chi tiết tại [AccuWeather](https://www.accuweather.com/vi/vn/hanoi/353412/hourly-weather-forecast/353412) hoặc [Thoitiet.vn](https://thoitiet.vn/ha-noi) để biết thêm thông tin về nhiệt độ, lượng mưa, độ ẩm và gió.'